# topk-predictions — ex2: top-1 vs top-5 contrast — count the wedge where top-5 hits but top-1 misses

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `topk-predictions`. Running the final beacon cell reports progress against the `Eval: topk predictions` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Eval: topk predictions` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`topk-predictions`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "topk-predictions"
DD_SUBTOPIC = "Eval: topk predictions"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## top-1 vs top-5 — when do they disagree?

Ex1 computed top-5 accuracy. The deepening move is to compute BOTH top-1 and top-5 on the same logits and surface the gap. Top-1 is strict (the argmax must hit the label); top-5 is permissive (the label must appear anywhere in the K=5 largest logits).

```python
top5 = logits.topk(5, dim=1).indices           # (B, 5)
top1 = logits.argmax(dim=1, keepdim=True)       # (B, 1)
hit5 = (top5 == labels[:, None]).any(dim=1)     # (B,) bool
hit1 = (top1.squeeze(1) == labels)              # (B,) bool
```

**`top5_only`** = samples where top-5 hit AND top-1 missed. That's the wedge between the two metrics — the model knows the label is in the running but isn't certain enough to commit. ImageNet ResNets famously have a ~7-point gap between top-1 and top-5; the wedge is where the model is hedging.

**Why `labels[:, None]` over `unsqueeze(1)`.** Same shape `(B, 1)`. Bracket-syntax is one fewer character and reads as 'add a column axis' instead of 'unsqueeze at position 1'.

### Exercise 2 — top-1 vs top-5 contrast — count the wedge where top-5 hits but top-1 misses

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply both `argmax` and `topk(5).indices` against the same labels and return three counts: top-1 hits, top-5 hits, and the wedge (top-5 hits but top-1 misses).
> Keywords: topk, top1, top5, accuracy
> ```

**KCs targeted:** `topk-membership-via-any`, `argmax-vs-topk-comparison`

Implement `ex2_top1_top5_wedge(logits, labels)`. Surfaces the gap between the two metrics on a single batch.

Inputs:
- `logits`: `(B, C)` float tensor.
- `labels`: `(B,)` long tensor with values in `[0, C)`.

Algorithm:
1. `top5 = logits.topk(5, dim=1).indices` — shape `(B, 5)`.
2. `top1 = logits.argmax(dim=1)` — shape `(B,)`.
3. `hit1 = (top1 == labels)` — shape `(B,)`, bool.
4. `hit5 = (top5 == labels[:, None]).any(dim=1)` — shape `(B,)`, bool. (`labels[:, None]` broadcasts the label across the 5 topk slots.)
5. `wedge = hit5 & (~hit1)` — top-5 captured the label but top-1 didn't. Return integers (Python ints, not tensors).

Output: `dict` with keys `'top1_hits'`, `'top5_hits'`, `'wedge'` — each a Python `int` count.

Constraint: assume `C >= 5`. If `C < 5`, `topk(5)` raises — that's PyTorch's contract, not yours to handle.

In [ ]:
def ex2_top1_top5_wedge(logits, labels):
    top5 = logits.topk(5, dim=1).indices
    top1 = logits.argmax(dim=1)
    hit1 = (top1 == labels)
    hit5 = (top5 == labels[:, None]).any(dim=1)
    wedge = hit5 & (~hit1)
    return {
        'top1_hits': int(hit1.sum().item()),
        'top5_hits': int(hit5.sum().item()),
        'wedge':     int(wedge.sum().item()),
    }


<details><summary>Solution</summary>

```python
def ex2_top1_top5_wedge(logits, labels):
    top5 = logits.topk(5, dim=1).indices
    top1 = logits.argmax(dim=1)
    hit1 = (top1 == labels)
    hit5 = (top5 == labels[:, None]).any(dim=1)
    wedge = hit5 & (~hit1)
    return {
        'top1_hits': int(hit1.sum().item()),
        'top5_hits': int(hit5.sum().item()),
        'wedge':     int(wedge.sum().item()),
    }
```

**Identity `wedge == top5_hits - top1_hits`.** Provable from the algebra: every top-1 hit is also a top-5 hit (the argmax is always in the top-K), so `hit1 ⊆ hit5`, and `wedge = hit5 ∧ ¬hit1 = hit5 - hit1`. Treat any deviation as a bug.

**`labels[:, None]` for the membership compare.** Reshape `(B,)` → `(B, 1)`, then broadcasts against `(B, 5)` from `topk`. Cleaner than `unsqueeze(1)`.

**`int(...)` over `.item()` alone.** `.item()` already returns a Python number; the outer `int(...)` is a defensive cast — `bool.sum().item()` can be a numpy int on some PyTorch builds. Forcing `int` makes downstream serialization (`json.dumps`) work uniformly.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()